# పాఠం 07 - ప్లానింగ్ డిజైన్ ప్యాటర్న్

ఈ నోట్‌బుక్ Microsoft Agent Framework ఉపయోగించి AI ఏజెంట్ల కోసం **ప్లానింగ్ డిజైన్ ప్యాటర్న్**ని చూపిస్తుంది.
మీరు క్లిష్టమైన ప్రయాణ అభ్యర్థనలను నిర్మిత ఉపపనుధ్యాయాలుగా విభజించడం, వాటిని నైపుణ్యం కలిగిన ఏజెంట్లకు కేటాయించడం నేర్చుకుంటారు,
మరియు Pydantic మోడల్స్ ద్వారాను శక్తివంతం చేయబడిన నిర్మిత ఔట్‌పుట్ ఉపయోగించి దృశ్యమైన ప్లాన్‌ను అమలు చేస్తారు.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## పనిని విడగొట్టడం  

ప్లానింగ్ డిజైన్ ప్యాటర్న్ యొక్క మూలభూత భాగం పనిని విడగొట్టడం. ఒకే ఏజెంట్‌ను  
ఒక సంక్లిష్ట అభ్యర్థనను ఎండ్-టు-ఎండ్ హ్యాండిల్ చేయమని అడుగుటకు బదులుగా, సమస్యను చిన్న, బాగా నిర్వచించిన **ఉపపనులను** విడదీస్తాము.  
ప్రతి ఉపపనికి ఒక నిపుణుడైన ఏజెంట్ (ఉదా: విమానాలు, హోటల్‌లు, కార్యకలాపాలు) కేటాయించి,  
స్పష్టమైన ప్రాధాన్యతలు మరియు ఆధారపు సరస్వతులను కలిగి ఉంటుంది.  

ఈ విధానంపైన కొన్ని లాభాలు ఉంటాయి:  
- **స్పష్టత**: ప్రతి ఉపపనికి ఒకే బాధ్యత ఉంటుంది.  
- **సమాంతరత్వం**: స్వతంత్ర ఉపపనులు సమాంతరంగా నడవచ్చు.  
- **నమ్మకదారితం**: విఫలమవకపోతే అది వ్యక్తుల ఉపపనిల వరకు పరిమితమవుతుంది.  
- **బడ్జెట్ ట్రాకింగ్**: వ్యయాలు ప్రతి ఉపపనికి అంచనా వేసి మొత్తంగా నమోదు చేయబడతాయి.  


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## నిర్మాణాత్మక అవుట్పుట్‌తో ఒక ప్లానింగ్ ఏజెంట్ సృష్టించడం

ప్లానింగ్ ఏజెంట్ ఒక **ఫ్రంట్ డెస్క్ కోఆర్డినేటర్**గా వ్యవహరిస్తుంది. ఒక ఉన్నత స్థాయి ప్రయాణ అభ్యర్థన ఇవ్వబడినప్పుడు,
ఇది నిర్మాణాత్మక `TravelPlan` ని ఉత్పత్తి చేస్తుంది — అభ్యర్థనను ఉపపనులకు విభజించటం, ప్రాధాన్యతలను ఏర్పాటు చేయడం,
మరియు కాన్సియర్జ్ లేదా అమలు పొర ఈ పని చేయగలుగునట్లు ఆధారాలను గుర్తించడం.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## నిపుణుల టూల్స్‌తో ప్లాన్‌ను అమలు చేయడం

ముందస్తు డెస్క్ ఏజెంట్ ఒక నిర్మితమైన ప్రణాళికను తయారు చేసిన వెంటనే, **కోన్సియర్ ఏజెంట్** దానిని అమలు చేస్తుంది.
ప్రతి నిపుణుల టూల్ ఒక ఉపఉద్యమ విభాగాన్ని నిర్వహిస్తుంది (ఫ్లైట్‌లు, హోటళ్లు, కార్యకలాపాలు). కోన్సియర్
ప్రణాళికలోని ఉపఉద్యమాలను ఆశ్రిత క్ర‌మంలో పునరావృతంగా పరిశీలించి ప్రతి ఒకటిని
సరైన టూల్‌కు పంపిస్తుంది.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

In this lesson you learned the **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — A front desk planning agent breaks a complex travel request into
   structured subtasks using Pydantic models, assigning each to a specialist agent with priorities
   and dependencies.
2. **Structured Output** — By passing a `response_format` the agent returns a validated
   `TravelPlan` object instead of free-form text, making downstream processing reliable.
3. **Plan Execution** — A concierge agent iterates through the subtasks using specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) to carry out the plan and report results.

This pattern separates *what to do* (planning) from *how to do it* (execution), making agents
more modular, testable, and easier to extend.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
